In [1]:
from google.colab import drive
# Mount Google Drive/
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import zipfile
import tarfile
import shutil

# List of zip files
zip_files = [
   "/content/drive/MyDrive/annotations_v2.zip",
   "/content/drive/MyDrive/data by hand.zip",
   "/content/drive/MyDrive/train_images.zip",
   "/content/drive/MyDrive/validation_images.zip",
   "/content/drive/MyDrive/dev_gold_labels.zip",
   "/content/drive/MyDrive/dev_images.zip"



]

# Iterate through each zip file
for zip_file in zip_files:
    # Extract the filename without extension
    file_name = os.path.splitext(os.path.basename(zip_file))[0]

    # Create a directory for each file
    extract_dir = os.path.join("/content", file_name)
    os.makedirs(extract_dir, exist_ok=True)

    try:
        # Check if the file is a zip archive
        if zipfile.is_zipfile(zip_file):
            with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                zip_ref.extractall(extract_dir)
                print(f"Extracted {zip_file} to {extract_dir}")
        # Check if the file is a tar archive
        elif tarfile.is_tarfile(zip_file):
            with tarfile.open(zip_file, 'r') as tar_ref:
                tar_ref.extractall(extract_dir)
                print(f"Extracted {zip_file} to {extract_dir}")
        else:
            print(f"Skipping {zip_file} as it is not a zip or tar archive")
    except Exception as e:
        print(f"Error extracting {zip_file}: {e}")

Extracted /content/drive/MyDrive/annotations_v2.zip to /content/annotations_v2
Extracted /content/drive/MyDrive/data by hand.zip to /content/data by hand
Extracted /content/drive/MyDrive/train_images.zip to /content/train_images
Extracted /content/drive/MyDrive/validation_images.zip to /content/validation_images
Extracted /content/drive/MyDrive/dev_gold_labels.zip to /content/dev_gold_labels
Extracted /content/drive/MyDrive/dev_images.zip to /content/dev_images


In [12]:
import json
import pandas as pd

# Load JSON file
with open('/content/drive/MyDrive/validation_caption.json', 'r') as f:
    data = json.load(f)

# Convert to DataFrame
df = pd.DataFrame(data)



# Display as table
display(df.head())



,id,text,image,labels,link,caption
0,63226,POLL: IF YOU THINK THIS MAN IS MENTALLY ILL\nL...,prop_meme_405.png,"[Loaded Language, Smears]",https://www.facebook.com/SilentmajorityDJT/pho...,This meme is a **political attack meme** targe...
1,64328,﻿FAKE NEWS PLANDEMIC\nVOTER FRAUD\nOPEN BORDER...,prop_meme_4269.png,"[Loaded Language, Slogans, Doubt, Flag-waving,...",https://www.facebook.com/photo/?fbid=102188324...,This meme presents a classic **conspiracy theo...
2,66831,I tell you\nwhat to wear.\nwhat to eat.\nwhat ...,prop_meme_5218.png,"[Repetition, Transfer, Black-and-white Fallacy...",null,This meme features a vintage television set wi...
3,79544,"IN CONGRESS FOR 30 YEARS\n$193,400 SALARY\n\nN...",prop_meme_24585.png,[Doubt],https://www.facebook.com/ResistanceFeed/photos...,Here's a factual evaluation of the claims made...
4,70781,NOLTE: BERNIE GETS A BRIEFING ABOUT RUSSIA\nME...,prop_meme_6607.png,"[Loaded Language, Smears, Whataboutism]",null,"This meme, attributed to ""Nolte"" (likely John ..."


In [13]:
print(f"The dataset has {len(df)} records.")

The dataset has 500 records.


In [4]:
import pandas as pd
import json

# Use the DataFrame 'df' loaded in the previous cell (QhWBjnkuTheA)
# Rename the column 'gemini_caption' to 'caption'
df.rename(columns={'gemini_caption': 'caption'}, inplace=True)

# Convert DataFrame to JSON format
# Use orient='records' to get a list of dictionaries, which is a common JSON format for tabular data
json_data = df.to_dict(orient='records')

# Define the output path for the JSON file
output_path = '/content/drive/MyDrive/validation_caption.json'

# Save the JSON data to a file
with open(output_path, 'w') as f:
    json.dump(json_data, f, indent=4) # Use indent for pretty printing

print(f"Updated DataFrame saved to: {output_path}")

Updated DataFrame saved to: /content/drive/MyDrive/validation_caption.json


In [ ]:
 {
    "Persuasion": ["Ethos", "Logos", "Pathos"],
    "Ethos": ["Ad Hominem", "Justification"],
    "Logos": ["Distraction", "Simplification"],
    "Pathos": ["Other", "Other"],
    "Ad Hominem": ["Name calling/Labeling", "Doubt", "Smears", "Reductio ad hitlerum", "Whataboutism"],
    "Justification": ["Flag-waving", "Appeal to fear/prejudice", "Bandwagon", "Slogans"],
    "Distraction": ["Misrepresentation of Someone's Position (Straw Man)", "Presenting Irrelevant Data (Red Herring)", "Whataboutism"],
    "Simplification": ["Black-and-white Fallacy/Dictatorship", "Thought-terminating cliché", "Causal Oversimplification"],
    "Other": ["Bandwagon", "Appeal to authority", "Glittering generalities (Virtue)", "Transfer", "Repetition",
             "Obfuscation, Intentional vagueness, Confusion", "Appeal to (Strong) Emotions", "Exaggeration/Minimisation",
             "Loaded Language", "Flag-waving", "Appeal to fear/prejudice", "Transfer"]
}

In [14]:
import os
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, precision_recall_curve
from sklearn.metrics.pairwise import cosine_similarity
#from sklearn.metrics import classification_report, precision_recall_curve, cosine_similarity
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from collections import Counter

from transformers import CLIPModel, CLIPProcessor, RobertaModel, RobertaTokenizer
import warnings
warnings.filterwarnings('ignore')

# Configuration
class CFG:
    # Paths - UPDATE THESE ACCORDING TO YOUR DATA STRUCTURE
    train_json = '/content/drive/MyDrive/dataset_with_rationales_subtask2a_final (1).json'
    val_json = '/content/drive/MyDrive/validation_caption.json'
    test_json = '/content/drive/MyDrive/dev_processed.json'

    train_img_dir = '/content/train_images/train_images'
    val_img_dir   = '/content/validation_images/validation_images'
    test_img_dir  = '/content/dev_images/dev_images'

    # Hyperparameters
    seed = 42
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    batch_size = 32
    lr = 2e-3
    epochs = 12
    validate_every = 100

    # Caption settings
    use_caption = True
    caption_separator = " [SEP] "

    # Model paths
    checkpoint_dir = './checkpoints'
    log_dir = './logs'

    # Model names
    clip_model_name = "openai/clip-vit-base-patch32"
    roberta_model_name = "roberta-base"

    # Training improvements
    gradient_clip_norm = 1.0
    early_stopping_patience = 5
    lr_scheduler_patience = 2
    lr_scheduler_factor = 0.5

    # GCN settings
    use_gcn = True
    gcn_hidden_dim = 256
    gcn_layers = 2
    pmi_weight = 0.6
    semantic_weight = 0.4

def set_seed(seed=42):
    """Set random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Hierarchical label structure
HIERARCHY_GRAPH = {
    "Persuasion": ["Ethos", "Logos", "Pathos"],
    "Ethos": ["Ad Hominem", "Justification"],
    "Logos": ["Distraction", "Simplification"],
    "Pathos": ["Other", "Other"],
    "Ad Hominem": ["Name calling/Labeling", "Doubt", "Smears", "Reductio ad hitlerum", "Whataboutism"],
    "Justification": ["Flag-waving", "Appeal to fear/prejudice", "Bandwagon", "Slogans"],
    "Distraction": ["Misrepresentation of Someone's Position (Straw Man)", "Presenting Irrelevant Data (Red Herring)", "Whataboutism"],
    "Simplification": ["Black-and-white Fallacy/Dictatorship", "Thought-terminating cliché", "Causal Oversimplification"],
    "Other": ["Bandwagon", "Appeal to authority", "Glittering generalities (Virtue)", "Transfer", "Repetition",
             "Obfuscation, Intentional vagueness, Confusion", "Appeal to (Strong) Emotions", "Exaggeration/Minimisation",
             "Loaded Language", "Flag-waving", "Appeal to fear/prejudice", "Transfer"]
}

# تعاریف هر propaganda technique
TECHNIQUE_DEFINITIONS = {
    "Name calling/Labeling": "Giving a person or idea a bad label to make the audience reject them without examining evidence",
    "Repetition": "Repeating the same message over and over again so that the audience will accept it",
    "Slogans": "A brief and striking phrase that contains labeling and stereotyping",
    "Appeal to fear/prejudice": "Seeking to build support by instilling anxiety and panic in the population",
    "Doubt": "Questioning the credibility of someone or something",
    "Exaggeration/Minimisation": "Either representing something in an excessive manner or making something seem less important",
    "Flag-waving": "Playing on strong national feeling to justify or promote an action",
    "Causal Oversimplification": "Assuming a single cause when there are multiple causes behind an issue",
    "Appeal to authority": "Supposing that a claim is true because a valid authority or expert on the issue said it",
    "Black-and-white Fallacy/Dictatorship": "Presenting two alternative options as the only possibilities",
    "Thought-terminating cliché": "Words or phrases that discourage critical thought and useful discussion",
    "Whataboutism": "Discredit an opponent's position by charging them with hypocrisy without refuting their argument",
    "Reductio ad hitlerum": "Comparing something/someone to Hitler or Nazism to make the argument seem invalid",
    "Bandwagon": "Attempting to persuade the target audience to join in and take the course of action because everyone else is doing so",
    "Obfuscation, Intentional vagueness, Confusion": "Using deliberately unclear words to make the message confusing",
    "Loaded Language": "Using specific words and phrases with strong emotional implications to influence the audience",
    "Glittering generalities (Virtue)": "Words associated with highly valued concepts that are used to evoke positive emotional response",
    "Misrepresentation of Someone's Position (Straw Man)": "When an opponent's proposition is substituted with a similar one",
    "Presenting Irrelevant Data (Red Herring)": "Introducing irrelevant material to the argument to distract",
    "Transfer": "Projecting positive or negative qualities of a person, entity, object to another",
    "Appeal to (Strong) Emotions": "Attempting to develop an emotional response instead of a valid or compelling argument",
    "Smears": "A direct attack on the reputation or character of a person or group",
}

def create_label_mappings():
    """Create label to index mappings and ancestor matrix"""
    all_labels = set(HIERARCHY_GRAPH.keys())
    for children in HIERARCHY_GRAPH.values():
        all_labels.update(children)

    label_to_idx = {label: i for i, label in enumerate(sorted(all_labels))}
    idx_to_label = {i: label for label, i in label_to_idx.items()}
    num_labels = len(label_to_idx)

    ancestors = {label_to_idx['Persuasion']: {label_to_idx['Persuasion']}}

    for parent, children in HIERARCHY_GRAPH.items():
        parent_idx = label_to_idx[parent]
        for child in children:
            child_idx = label_to_idx[child]
            ancestors[child_idx] = ancestors.get(child_idx, set()) | ancestors.get(parent_idx, set()) | {child_idx}

    ancestor_matrix = torch.zeros((num_labels, num_labels), dtype=torch.float32)
    for node, anc_set in ancestors.items():
        ancestor_matrix[node, list(anc_set)] = 1.0

    return label_to_idx, idx_to_label, ancestor_matrix, num_labels

def compute_pmi_matrix(labels_data, label_to_idx, smooth=1e-5):
    """محاسبه PMI (Pointwise Mutual Information) matrix از داده‌های training"""
    num_labels = len(label_to_idx)
    co_occurrence = np.zeros((num_labels, num_labels))
    label_counts = np.zeros(num_labels)
    total_samples = len(labels_data)

    print(f"Computing PMI from {total_samples} training samples...")

    # شمارش co-occurrences
    for labels in tqdm(labels_data, desc="Computing co-occurrences"):
        label_indices = [label_to_idx[label] for label in labels if label in label_to_idx]

        # شمارش تک‌تک label‌ها
        for idx in label_indices:
            label_counts[idx] += 1

        # شمارش co-occurrences
        for i in label_indices:
            for j in label_indices:
                co_occurrence[i, j] += 1

    # محاسبه PMI
    pmi_matrix = np.zeros((num_labels, num_labels))

    for i in range(num_labels):
        for j in range(num_labels):
            p_i = (label_counts[i] + smooth) / total_samples
            p_j = (label_counts[j] + smooth) / total_samples
            p_ij = (co_occurrence[i, j] + smooth) / total_samples

            pmi = np.log(p_ij / (p_i * p_j))
            pmi_matrix[i, j] = max(0, pmi)  # فقط PMI مثبت

    # نرمال‌سازی به بازه [0, 1]
    if pmi_matrix.max() > 0:
        pmi_matrix = pmi_matrix / pmi_matrix.max()

    return pmi_matrix

def compute_semantic_similarity(label_to_idx, idx_to_label, roberta_model, roberta_tokenizer, device):
    """محاسبه semantic similarity بین تعاریف techniques با RoBERTa"""
    num_labels = len(label_to_idx)
    similarity_matrix = np.zeros((num_labels, num_labels))

    print("Computing semantic similarity from technique definitions...")

    # استخراج embeddings برای هر تعریف
    embeddings = []

    with torch.no_grad():
        for i in tqdm(range(num_labels), desc="Extracting embeddings"):
            label_name = idx_to_label[i]
            definition = TECHNIQUE_DEFINITIONS.get(label_name, label_name)

            encoded = roberta_tokenizer(
                definition,
                padding='max_length',
                truncation=True,
                max_length=128,
                return_tensors='pt'
            )

            input_ids = encoded['input_ids'].to(device)
            attention_mask = encoded['attention_mask'].to(device)

            outputs = roberta_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
            embeddings.append(embedding)

    embeddings = np.array(embeddings)

    # محاسبه cosine similarity
    similarity_matrix = cosine_similarity(embeddings)

    # نرمال‌سازی به بازه [0, 1]
    similarity_matrix = (similarity_matrix + 1) / 2  # از [-1, 1] به [0, 1]

    return similarity_matrix

def build_label_graph(train_df, label_to_idx, idx_to_label, roberta_model, roberta_tokenizer, device,
                      pmi_weight=0.6, semantic_weight=0.4):
    """ساخت adjacency matrix ترکیبی از PMI و semantic similarity"""

    print("\n" + "="*60)
    print("BUILDING LABEL GRAPH FOR GCN")
    print("="*60)

    # محاسبه PMI
    labels_data = train_df['labels'].tolist()
    pmi_matrix = compute_pmi_matrix(labels_data, label_to_idx)
    print(f"✓ PMI matrix computed: {pmi_matrix.shape}")
    print(f"  Non-zero entries: {np.count_nonzero(pmi_matrix)}")
    print(f"  Average PMI: {pmi_matrix.mean():.4f}")

    # محاسبه semantic similarity
    semantic_matrix = compute_semantic_similarity(label_to_idx, idx_to_label,
                                                   roberta_model, roberta_tokenizer, device)
    print(f"✓ Semantic similarity matrix computed: {semantic_matrix.shape}")
    print(f"  Average similarity: {semantic_matrix.mean():.4f}")

    # ترکیب دو ماتریس
    adjacency_matrix = pmi_weight * pmi_matrix + semantic_weight * semantic_matrix

    # اضافه کردن self-loops
    np.fill_diagonal(adjacency_matrix, 1.0)

    # نرمال‌سازی ماتریس (symmetric normalization)
    # A_norm = D^(-1/2) * A * D^(-1/2)
    degree = adjacency_matrix.sum(axis=1)
    degree_inv_sqrt = np.power(degree, -0.5)
    degree_inv_sqrt[np.isinf(degree_inv_sqrt)] = 0.0
    degree_matrix_inv_sqrt = np.diag(degree_inv_sqrt)

    adjacency_matrix_norm = degree_matrix_inv_sqrt @ adjacency_matrix @ degree_matrix_inv_sqrt

    print(f"✓ Combined adjacency matrix created:")
    print(f"  PMI weight: {pmi_weight}, Semantic weight: {semantic_weight}")
    print(f"  Matrix shape: {adjacency_matrix_norm.shape}")
    print(f"  Non-zero entries: {np.count_nonzero(adjacency_matrix_norm)}")
    print(f"  Density: {np.count_nonzero(adjacency_matrix_norm) / (adjacency_matrix_norm.shape[0] ** 2):.4f}")
    print("="*60)

    return torch.FloatTensor(adjacency_matrix_norm)

class FocalLoss(nn.Module):
    """Focal Loss for addressing class imbalance"""
    def __init__(self, alpha=1.0, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Calculate BCE loss
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')

        # Calculate pt
        pt = torch.exp(-bce_loss)

        # Calculate focal loss
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

class EarlyStopping:
    """Early stopping to prevent overfitting"""
    def __init__(self, patience=7, min_delta=0.001, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_score = None
        self.counter = 0
        self.best_weights = None
        self.early_stop = False

    def __call__(self, score, model):
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(model)
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                if self.restore_best_weights:
                    model.load_state_dict(self.best_weights)
        else:
            self.best_score = score
            self.counter = 0
            self.save_checkpoint(model)

    def save_checkpoint(self, model):
        """Save model when validation score improves"""
        self.best_weights = model.state_dict().copy()

class GraphConvolutionLayer(nn.Module):
    """یک لایه Graph Convolution"""

    def __init__(self, in_features, out_features, bias=True):
        super(GraphConvolutionLayer, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))

        if bias:
            self.bias = nn.Parameter(torch.FloatTensor(out_features))
        else:
            self.register_parameter('bias', None)

        self.reset_parameters()

    def reset_parameters(self):
        """Initialize parameters"""
        nn.init.xavier_uniform_(self.weight)
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def forward(self, input_features, adjacency_matrix):
        """
        Args:
            input_features: (batch_size, num_labels, in_features)
            adjacency_matrix: (num_labels, num_labels)
        Returns:
            output: (batch_size, num_labels, out_features)
        """
        # Linear transformation: XW
        support = torch.matmul(input_features, self.weight)

        # Graph convolution: AXW
        # support: [batch_size, num_labels, out_features]
        # adjacency_matrix: [num_labels, num_labels]
        # نیاز به: output[b, i, f] = sum_j(adjacency_matrix[i, j] * support[b, j, f])
        output = torch.einsum('ij,bjf->bif', adjacency_matrix, support)

        if self.bias is not None:
            output = output + self.bias

        return output

class LabelGCN(nn.Module):
    """Graph Convolutional Network برای refine کردن predictions"""

    def __init__(self, num_labels, hidden_dim=256, num_layers=2, dropout=0.3):
        super(LabelGCN, self).__init__()

        self.num_labels = num_labels
        self.num_layers = num_layers

        # GCN layers
        self.gcn_layers = nn.ModuleList()

        # اولین لایه: از 1 feature (prediction score) به hidden_dim
        self.gcn_layers.append(GraphConvolutionLayer(1, hidden_dim))

        # لایه‌های میانی
        for _ in range(num_layers - 2):
            self.gcn_layers.append(GraphConvolutionLayer(hidden_dim, hidden_dim))

        # آخرین لایه: از hidden_dim به 1 (refined prediction)
        if num_layers > 1:
            self.gcn_layers.append(GraphConvolutionLayer(hidden_dim, 1))

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.batch_norm = nn.ModuleList([nn.BatchNorm1d(hidden_dim) for _ in range(num_layers - 1)])

        # Residual connection weight - learnable parameter
        self.residual_weight = nn.Parameter(torch.FloatTensor([0.5]))

    def forward(self, predictions, adjacency_matrix):
        """
        Args:
            predictions: (batch_size, num_labels) - initial predictions
            adjacency_matrix: (num_labels, num_labels) - label graph
        Returns:
            refined_predictions: (batch_size, num_labels) - refined predictions
        """
        batch_size = predictions.size(0)

        # تبدیل predictions به (batch_size, num_labels, 1)
        x = predictions.unsqueeze(-1)

        # عبور از لایه‌های GCN
        for i, gcn_layer in enumerate(self.gcn_layers):
            x = gcn_layer(x, adjacency_matrix)

            # ReLU, BatchNorm و Dropout برای همه لایه‌ها به جز آخری
            if i < len(self.gcn_layers) - 1:
                # Reshape for batch norm: (batch_size, num_labels, hidden_dim) -> (batch_size, hidden_dim, num_labels)
                x = x.transpose(1, 2)
                x = self.batch_norm[i](x)
                x = x.transpose(1, 2)

                x = self.relu(x)
                x = self.dropout(x)

        # برگرداندن به شکل (batch_size, num_labels)
        refined = x.squeeze(-1)

        # Residual connection: ترکیب predictions اولیه با refined predictions
        # استفاده از sigmoid برای محدود کردن residual_weight به [0, 1]
        alpha = torch.sigmoid(self.residual_weight)
        output = alpha * predictions + (1 - alpha) * refined

        return output

class MemeDataset(Dataset):
    """Dataset class for meme classification with caption support"""

    def __init__(self, df, img_dir, processor, label_to_idx, ancestor_matrix,
                 is_test=False, use_caption=True, caption_separator=" [SEP] "):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.processor = processor
        self.label_to_idx = label_to_idx
        self.ancestor_matrix = ancestor_matrix
        self.num_labels = len(label_to_idx)
        self.is_test = is_test
        self.use_caption = use_caption
        self.caption_separator = caption_separator

    def __len__(self):
        return len(self.df)

    def _encode_labels(self, label_list):
        """Convert label list to one-hot vector with hierarchical expansion"""
        if not label_list:
            return torch.zeros(self.num_labels)

        label_indices = [self.label_to_idx[label] for label in label_list if label in self.label_to_idx]

        expanded_indices = set()
        for idx in label_indices:
            ancestors = torch.where(self.ancestor_matrix[idx] == 1)[0].tolist()
            expanded_indices.update(ancestors)

        y = torch.zeros(self.num_labels)
        y[list(expanded_indices)] = 1.0
        return y

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Get text
        text = row['text'] if 'text' in row and pd.notna(row['text']) else ""

        # Get caption if available and combine with text
        if self.use_caption and 'caption' in row and pd.notna(row['caption']):
            caption = row['caption']
            combined_text = f"{text}{self.caption_separator}{caption}"
        else:
            combined_text = text

        # Handle image loading
        image = None
        if 'image' in row:
            img_path = self.img_dir / row['image']
            try:
                image = Image.open(img_path).convert('RGB')
            except Exception as e:
                if not str(img_path).startswith('path/to/'):
                    print(f"Error loading image {img_path}: {e}")
                image = None

        if image is None:
            colors = ['white', 'lightgray', 'lightblue', 'lightgreen', 'lightyellow', 'lightpink']
            color = random.choice(colors)
            image = Image.new('RGB', (224, 224), color=color)

        # Encode labels
        if self.is_test or 'labels' not in row:
            labels = torch.zeros(self.num_labels)
        else:
            labels = self._encode_labels(row['labels'])

        return combined_text, image, labels, idx

class ImprovedMultiHeadMLP(nn.Module):
    """Improved Multi-Head MLP with GCN refinement and residual connections"""

    def __init__(self, input_dim=1792, num_labels=22, use_gcn=True, gcn_hidden_dim=256, gcn_layers=2):
        super(ImprovedMultiHeadMLP, self).__init__()

        self.use_gcn = use_gcn

        # === HEAD 1 DEDICATED LAYERS ===
        self.layer1 = nn.Linear(input_dim, 768)
        self.layer1_bn = nn.BatchNorm1d(768)

        self.layer2 = nn.Linear(768, 512)
        self.layer2_bn = nn.BatchNorm1d(512)

        # Head 1 output (Ethos, Pathos, Logos)
        self.head1 = nn.Linear(512, 3)

        # Head 1 feature reduction for final head
        self.head1_reducer = nn.Linear(512, 64)

        # Residual connection from head1 to head2
        self.head1_to_head2_residual = nn.Linear(512, 128)

        # === HEAD 2 DEDICATED LAYERS ===
        self.layer3 = nn.Linear(512, 256)
        self.layer3_bn = nn.BatchNorm1d(256)

        self.layer4 = nn.Linear(256, 128)
        self.layer4_bn = nn.BatchNorm1d(128)

        # Head 2 output
        self.head2 = nn.Linear(128, 5)

        # Head 2 feature reduction for final head
        self.head2_reducer = nn.Linear(128, 64)

        # Residual connection from head2 to final
        self.head2_to_final_residual = nn.Linear(128, 64)

        # === HEAD 3 (FINAL) LAYERS ===
        self.final_layer1 = nn.Linear(128, 96)
        self.final_layer1_bn = nn.BatchNorm1d(96)

        self.final_layer2 = nn.Linear(96, 64)
        self.final_layer2_bn = nn.BatchNorm1d(64)

        # Additional residual connection in final head
        self.final_residual = nn.Linear(128, 64)

        self.final_head = nn.Linear(64, num_labels)

        # === GCN برای refinement ===
        if self.use_gcn:
            self.gcn = LabelGCN(
                num_labels=num_labels,
                hidden_dim=gcn_hidden_dim,
                num_layers=gcn_layers,
                dropout=0.3
            )
            print(f"✓ GCN initialized with {gcn_layers} layers and hidden_dim={gcn_hidden_dim}")

        # Activation and dropout
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x, adjacency_matrix=None, training_phase='all'):
        """Forward pass with GCN refinement and residual connections"""

        # === HEAD 1 PATH ===
        x1 = self.relu(self.layer1_bn(self.layer1(x)))
        x1 = self.dropout(x1)

        x1 = self.relu(self.layer2_bn(self.layer2(x1)))
        x1 = self.dropout(x1)

        # Head 1 output
        output1 = self.head1(x1)

        # Head 1 features for final head
        head1_features = self.relu(self.head1_reducer(x1))
        head1_features = self.dropout(head1_features)

        # === HEAD 2 PATH ===
        if training_phase in ['head2', 'head3', 'all']:
            if training_phase == 'head2':
                x2_input = x1.detach()
            else:
                x2_input = x1

            x2 = self.relu(self.layer3_bn(self.layer3(x2_input)))
            x2 = self.dropout(x2)

            x2 = self.relu(self.layer4_bn(self.layer4(x2)))
            x2 = self.dropout(x2)

            # Add residual connection from head1
            head1_residual = self.head1_to_head2_residual(x1)
            if training_phase == 'head2':
                head1_residual = head1_residual.detach()
            x2 = x2 + head1_residual

            # Head 2 output
            output2 = self.head2(x2)

            # Head 2 features for final head
            head2_features = self.relu(self.head2_reducer(x2))
            head2_features = self.dropout(head2_features)
        else:
            output2 = torch.zeros(x.size(0), 5, device=x.device)
            head2_features = torch.zeros(x.size(0), 64, device=x.device)
            x2 = torch.zeros(x.size(0), 128, device=x.device)

        # === HEAD 3 (FINAL) PATH ===
        if training_phase in ['head3', 'all']:
            if training_phase == 'head3':
                combined_features = torch.cat([head1_features.detach(), head2_features.detach()], dim=1)
                final_residual_input = torch.cat([head1_features.detach(), head2_features.detach()], dim=1)
            else:
                combined_features = torch.cat([head1_features, head2_features], dim=1)
                final_residual_input = combined_features

            # Final head layers with residual
            x3 = self.relu(self.final_layer1_bn(self.final_layer1(combined_features)))
            x3 = self.dropout(x3)

            x3 = self.relu(self.final_layer2_bn(self.final_layer2(x3)))
            x3 = self.dropout(x3)

            # Add residual connection in final head
            final_residual = self.final_residual(final_residual_input)
            x3 = x3 + final_residual

            # Final output (before GCN)
            output_final = self.final_head(x3)

            # === GCN REFINEMENT ===
            if self.use_gcn and adjacency_matrix is not None:
                # Refine predictions using label graph
                output_final_refined = self.gcn(output_final, adjacency_matrix)
                return output1, output2, output_final_refined
            else:
                return output1, output2, output_final
        else:
            output_final = torch.zeros(x.size(0), self.final_head.out_features, device=x.device)
            return output1, output2, output_final

    def freeze_head1_layers(self):
        """Freeze Head 1 specific layers"""
        for param in [self.layer1.parameters(), self.layer1_bn.parameters(),
                     self.layer2.parameters(), self.layer2_bn.parameters(),
                     self.head1.parameters(), self.head1_reducer.parameters(),
                     self.head1_to_head2_residual.parameters()]:
            for p in param:
                p.requires_grad = False

    def unfreeze_head1_layers(self):
        """Unfreeze Head 1 specific layers"""
        for param in [self.layer1.parameters(), self.layer1_bn.parameters(),
                     self.layer2.parameters(), self.layer2_bn.parameters(),
                     self.head1.parameters(), self.head1_reducer.parameters(),
                     self.head1_to_head2_residual.parameters()]:
            for p in param:
                p.requires_grad = True

    def freeze_head2_layers(self):
        """Freeze Head 2 specific layers"""
        for param in [self.layer3.parameters(), self.layer3_bn.parameters(),
                     self.layer4.parameters(), self.layer4_bn.parameters(),
                     self.head2.parameters(), self.head2_reducer.parameters(),
                     self.head2_to_final_residual.parameters()]:
            for p in param:
                p.requires_grad = False

    def unfreeze_head2_layers(self):
        """Unfreeze Head 2 specific layers"""
        for param in [self.layer3.parameters(), self.layer3_bn.parameters(),
                     self.layer4.parameters(), self.layer4_bn.parameters(),
                     self.head2.parameters(), self.head2_reducer.parameters(),
                     self.head2_to_final_residual.parameters()]:
            for p in param:
                p.requires_grad = True

    def freeze_head3_layers(self):
        """Freeze Head 3 and GCN layers"""
        for param in [self.final_layer1.parameters(), self.final_layer1_bn.parameters(),
                     self.final_layer2.parameters(), self.final_layer2_bn.parameters(),
                     self.final_head.parameters(), self.final_residual.parameters()]:
            for p in param:
                p.requires_grad = False

        if self.use_gcn:
            for param in self.gcn.parameters():
                param.requires_grad = False

    def unfreeze_head3_layers(self):
        """Unfreeze Head 3 and GCN layers"""
        for param in [self.final_layer1.parameters(), self.final_layer1_bn.parameters(),
                     self.final_layer2.parameters(), self.final_layer2_bn.parameters(),
                     self.final_head.parameters(), self.final_residual.parameters()]:
            for p in param:
                p.requires_grad = True

        if self.use_gcn:
            for param in self.gcn.parameters():
                param.requires_grad = True

def hierarchical_f1_score(y_pred_logits, y_true, ancestor_matrix, threshold=0.0, beta=1.0):
    """Calculate hierarchical F1 score"""
    y_pred = (y_pred_logits > threshold).float()

    y_true_expanded = torch.clamp(y_true @ ancestor_matrix, 0, 1)
    y_pred_expanded = torch.clamp(y_pred @ ancestor_matrix, 0, 1)

    tp = (y_true_expanded * y_pred_expanded).sum()

    true_sum = y_true_expanded.sum()
    pred_sum = y_pred_expanded.sum()

    if true_sum == 0 and pred_sum == 0:
        return 1.0, 1.0, 1.0
    elif true_sum == 0:
        return 0.0, 0.0, 0.0
    elif pred_sum == 0:
        return 0.0, 0.0, 0.0

    hierarchical_recall = tp / true_sum
    hierarchical_precision = tp / pred_sum

    if hierarchical_recall + hierarchical_precision == 0:
        hierarchical_f1 = 0.0
    else:
        hierarchical_f1 = (1 + beta**2) * hierarchical_recall * hierarchical_precision / \
                          (hierarchical_recall + beta**2 * hierarchical_precision)

    return hierarchical_f1.item(), hierarchical_recall.item(), hierarchical_precision.item()

def hierarchical_consistency_loss(predictions, ancestor_matrix, lambda_consistency=0.1):
    """Calculate hierarchical consistency loss"""
    # Apply sigmoid to get probabilities
    probs = torch.sigmoid(predictions)

    # For each sample, ensure hierarchy consistency
    # If a child is predicted, all ancestors should have higher or equal probability
    batch_size, num_labels = probs.shape

    consistency_violations = 0
    total_pairs = 0

    for i in range(num_labels):
        # Find all ancestors of label i
        ancestors = torch.where(ancestor_matrix[i] == 1)[0]

        for ancestor in ancestors:
            if ancestor != i:  # Skip self
                # Child probability should not exceed ancestor probability
                violation = torch.relu(probs[:, i] - probs[:, ancestor])
                consistency_violations += violation.sum()
                total_pairs += batch_size

    if total_pairs > 0:
        avg_violation = consistency_violations / total_pairs
        return lambda_consistency * avg_violation
    else:
        return torch.tensor(0.0, device=predictions.device, requires_grad=True)

def compute_class_weights(train_dataset, label_to_idx):
    """Compute class weights for handling imbalance"""
    class_counts = torch.zeros(len(label_to_idx))

    for i in range(len(train_dataset)):
        _, _, labels, _ = train_dataset[i]
        class_counts += labels

    # Avoid division by zero
    class_counts = torch.clamp(class_counts, min=1)

    # Inverse frequency weighting
    total_samples = class_counts.sum()
    class_weights = total_samples / (len(label_to_idx) * class_counts)

    return class_weights

def find_optimal_thresholds(y_true, y_pred_probs, num_classes):
    """Find optimal threshold for each class using F1 score"""
    optimal_thresholds = np.zeros(num_classes)

    for i in range(num_classes):
        if y_true[:, i].sum() > 0:  # Only if class has positive samples
            precision, recall, thresholds = precision_recall_curve(y_true[:, i], y_pred_probs[:, i])

            # Calculate F1 scores
            f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)

            # Find threshold that maximizes F1
            best_threshold_idx = np.argmax(f1_scores)
            if best_threshold_idx < len(thresholds):
                optimal_thresholds[i] = thresholds[best_threshold_idx]
            else:
                optimal_thresholds[i] = 0.5
        else:
            optimal_thresholds[i] = 0.5

    return optimal_thresholds

class ImprovedMemeClassifier:
    """Improved classifier with focal loss, residual connections, GCN and advanced training"""

    def __init__(self, config, train_df=None):
        self.config = config
        set_seed(config.seed)

        os.makedirs(config.checkpoint_dir, exist_ok=True)
        os.makedirs(config.log_dir, exist_ok=True)

        self.label_to_idx, self.idx_to_label, self.ancestor_matrix, self.num_labels = create_label_mappings()

        # Initialize RoBERTa for text encoding
        print("Loading RoBERTa model...")
        self.roberta_model = RobertaModel.from_pretrained(config.roberta_model_name).to(config.device)
        self.roberta_tokenizer = RobertaTokenizer.from_pretrained(config.roberta_model_name)

        # Freeze RoBERTa parameters
        for param in self.roberta_model.parameters():
            param.requires_grad = False
        self.roberta_model.eval()
        print("✓ RoBERTa model loaded and frozen")

        # Initialize CLIP
        print("Loading CLIP model...")
        self.clip_model = CLIPModel.from_pretrained(config.clip_model_name).to(config.device)
        self.clip_processor = CLIPProcessor.from_pretrained(config.clip_model_name)

        # Freeze CLIP parameters
        for param in self.clip_model.parameters():
            param.requires_grad = False
        self.clip_model.eval()
        print("✓ CLIP model loaded and frozen")

        # Initialize Improved Multi-Head MLP with GCN
        self.classifier = ImprovedMultiHeadMLP(
            input_dim=1792,
            num_labels=self.num_labels,
            use_gcn=config.use_gcn,
            gcn_hidden_dim=config.gcn_hidden_dim,
            gcn_layers=config.gcn_layers
        ).to(config.device)

        # ساخت label graph اگر train_df موجود باشد
        if train_df is not None and config.use_gcn:
            self.adjacency_matrix_graph = build_label_graph(
                train_df=train_df,
                label_to_idx=self.label_to_idx,
                idx_to_label=self.idx_to_label,
                roberta_model=self.roberta_model,
                roberta_tokenizer=self.roberta_tokenizer,
                device=self.config.device,
                pmi_weight=config.pmi_weight,
                semantic_weight=config.semantic_weight
            ).to(self.config.device)
        else:
            self.adjacency_matrix_graph = None
            if config.use_gcn:
                print("⚠ No training data provided - GCN will not be used")

        # Separate optimizers for each head
        self.optimizer_head1 = Adam([
            *self.classifier.layer1.parameters(),
            *self.classifier.layer1_bn.parameters(),
            *self.classifier.layer2.parameters(),
            *self.classifier.layer2_bn.parameters(),
            *self.classifier.head1.parameters(),
            *self.classifier.head1_reducer.parameters(),
            *self.classifier.head1_to_head2_residual.parameters()
        ], lr=config.lr)

        self.optimizer_head2 = Adam([
            *self.classifier.layer3.parameters(),
            *self.classifier.layer3_bn.parameters(),
            *self.classifier.layer4.parameters(),
            *self.classifier.layer4_bn.parameters(),
            *self.classifier.head2.parameters(),
            *self.classifier.head2_reducer.parameters(),
            *self.classifier.head2_to_final_residual.parameters()
        ], lr=config.lr)

        # اضافه کردن GCN parameters به optimizer head3
        head3_params = [
            *self.classifier.final_layer1.parameters(),
            *self.classifier.final_layer1_bn.parameters(),
            *self.classifier.final_layer2.parameters(),
            *self.classifier.final_layer2_bn.parameters(),
            *self.classifier.final_head.parameters(),
            *self.classifier.final_residual.parameters()
        ]

        if self.classifier.use_gcn:
            head3_params.extend(self.classifier.gcn.parameters())

        self.optimizer_head3 = Adam(head3_params, lr=config.lr)

        # Learning rate schedulers
        self.scheduler_head1 = ReduceLROnPlateau(
            self.optimizer_head1,
            patience=config.lr_scheduler_patience,
            factor=config.lr_scheduler_factor
        )
        self.scheduler_head2 = ReduceLROnPlateau(
            self.optimizer_head2,
            patience=config.lr_scheduler_patience,
            factor=config.lr_scheduler_factor
        )
        self.scheduler_head3 = ReduceLROnPlateau(
            self.optimizer_head3,
            patience=config.lr_scheduler_patience,
            factor=config.lr_scheduler_factor
        )

        # Loss functions
        self.focal_loss = FocalLoss(alpha=1.0, gamma=2.0)
        self.bce_loss = nn.BCEWithLogitsLoss()

        self.ancestor_matrix = self.ancestor_matrix.to(config.device)

        # Feature caches
        self.feature_cache = {}

        # Early stopping
        self.early_stopping = EarlyStopping(patience=config.early_stopping_patience)

        # Optimal thresholds (will be computed during validation)
        self.optimal_thresholds = None

        self.history = {
            'train_loss': [],
            'val_loss': [],
            'val_f1': [],
            'val_precision': [],
            'val_recall': [],
            'test_loss': [],
            'test_f1': [],
            'test_precision': [],
            'test_recall': []
        }

    def extract_roberta_features(self, texts):
        """Extract RoBERTa features from text+caption"""
        with torch.no_grad():
            encoded = self.roberta_tokenizer(
                texts,
                padding='max_length',
                truncation=True,
                max_length=256,
                return_tensors='pt'
            )

            input_ids = encoded['input_ids'].to(self.config.device)
            attention_mask = encoded['attention_mask'].to(self.config.device)

            outputs = self.roberta_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            roberta_features = outputs.last_hidden_state[:, 0, :]
            return roberta_features

    def extract_clip_features(self, texts, images):
        """Extract CLIP features from text and images"""
        with torch.no_grad():
            inputs = self.clip_processor(
                text=texts,
                images=images,
                return_tensors='pt',
                padding='max_length',
                truncation=True,
                max_length=77
            )

            for key in inputs:
                inputs[key] = inputs[key].to(self.config.device)

            outputs = self.clip_model(**inputs)
            clip_features = torch.cat((outputs.text_embeds, outputs.image_embeds), dim=-1)
            return clip_features

    def extract_features(self, texts, images):
        """Extract combined RoBERTa + CLIP features"""
        roberta_features = self.extract_roberta_features(list(texts))
        clip_features = self.extract_clip_features(list(texts), list(images))
        combined_features = torch.cat((roberta_features, clip_features), dim=-1)
        return combined_features

    def precompute_features(self, dataset, cache_name, batch_size=32):
        """Precompute and cache features for a dataset"""
        print(f"\n{'='*60}")
        print(f"Pre-computing features for {cache_name}...")
        print(f"{'='*60}")

        features_list = []
        labels_list = []

        num_samples = len(dataset)
        num_batches = (num_samples + batch_size - 1) // batch_size

        with torch.no_grad():
            for batch_idx in tqdm(range(num_batches), desc=f"Extracting {cache_name} features"):
                start_idx = batch_idx * batch_size
                end_idx = min(start_idx + batch_size, num_samples)

                batch_texts = []
                batch_images = []
                batch_labels = []

                for idx in range(start_idx, end_idx):
                    text, image, label, _ = dataset[idx]
                    batch_texts.append(text)
                    batch_images.append(image)
                    batch_labels.append(label)

                features = self.extract_features(batch_texts, batch_images)
                labels_tensor = torch.stack(batch_labels)

                features_list.append(features.cpu())
                labels_list.append(labels_tensor.cpu())

        all_features = torch.cat(features_list, dim=0)
        all_labels = torch.cat(labels_list, dim=0)

        self.feature_cache[cache_name] = {
            'features': all_features,
            'labels': all_labels
        }

        print(f"✓ Cached {len(all_features)} feature vectors for {cache_name}")
        print(f"  Feature shape: {all_features.shape}")

        return all_features, all_labels

    def collate_fn_cached(self, batch):
        """Fast collate function using cached features"""
        indices = [item[3] for item in batch]
        cache_name = getattr(self, '_current_cache', None)

        if cache_name and cache_name in self.feature_cache:
            cached_data = self.feature_cache[cache_name]
            features = cached_data['features'][indices]
            labels = cached_data['labels'][indices]
        else:
            texts, images, labels_list, _ = zip(*batch)
            labels = torch.stack(labels_list)
            features = self.extract_features(list(texts), list(images))

        return features.to(self.config.device), labels.to(self.config.device), torch.tensor(indices)

    def create_hierarchical_targets(self, labels):
        """Create targets for different heads"""
        batch_size = labels.shape[0]

        # Head 1 targets (Ethos, Pathos, Logos)
        ethos_idx = self.label_to_idx['Ethos']
        pathos_idx = self.label_to_idx['Pathos']
        logos_idx = self.label_to_idx['Logos']

        head1_targets = torch.zeros(batch_size, 3)
        head1_targets[:, 0] = labels[:, ethos_idx]
        head1_targets[:, 1] = labels[:, pathos_idx]
        head1_targets[:, 2] = labels[:, logos_idx]

        # Head 2 targets (Ad Hominem, Justification, Distraction, Simplification, Other)
        ad_hominem_idx = self.label_to_idx['Ad Hominem']
        justification_idx = self.label_to_idx['Justification']
        distraction_idx = self.label_to_idx['Distraction']
        simplification_idx = self.label_to_idx['Simplification']
        other_idx = self.label_to_idx['Other']

        head2_targets = torch.zeros(batch_size, 5)
        head2_targets[:, 0] = labels[:, ad_hominem_idx]
        head2_targets[:, 1] = labels[:, justification_idx]
        head2_targets[:, 2] = labels[:, distraction_idx]
        head2_targets[:, 3] = labels[:, simplification_idx]
        head2_targets[:, 4] = labels[:, other_idx]

        return head1_targets, head2_targets

    def create_progressive_class_masks(self, train_dataset):
        """Create class masks for progressive training"""
        # Count class frequencies
        class_counts = torch.zeros(self.num_labels)
        for i in range(len(train_dataset)):
            _, _, labels, _ = train_dataset[i]
            class_counts += labels

        # Define frequency thresholds
        high_freq_threshold = 100  # Classes with >100 samples
        med_freq_threshold = 20    # Classes with 20-100 samples

        # Create masks
        high_freq_mask = class_counts >= high_freq_threshold
        med_freq_mask = class_counts >= med_freq_threshold
        all_classes_mask = torch.ones(self.num_labels, dtype=torch.bool)

        print(f"High frequency classes: {high_freq_mask.sum().item()}")
        print(f"Medium+ frequency classes: {med_freq_mask.sum().item()}")
        print(f"All classes: {all_classes_mask.sum().item()}")

        return high_freq_mask, med_freq_mask, all_classes_mask

    def train_phase_progressive(self, train_loader, phase, class_mask, epochs_for_phase=3):
        """Train a specific phase with GCN support and class masking for progressive training"""
        print(f"\n{'='*60}")
        print(f"PROGRESSIVE TRAINING PHASE: {phase.upper()}")
        print(f"Training {class_mask.sum().item()} classes for {epochs_for_phase} epochs")
        if phase == 'head3' and self.classifier.use_gcn and self.adjacency_matrix_graph is not None:
            print("  ✓ GCN enabled for label refinement")
        print(f"{'='*60}")

        # Set up training for specific phase
        if phase == 'head1':
            self.classifier.freeze_head2_layers()
            self.classifier.freeze_head3_layers()
            self.classifier.unfreeze_head1_layers()
            optimizer = self.optimizer_head1
            scheduler = self.scheduler_head1
        elif phase == 'head2':
            self.classifier.freeze_head1_layers()
            self.classifier.freeze_head3_layers()
            self.classifier.unfreeze_head2_layers()
            optimizer = self.optimizer_head2
            scheduler = self.scheduler_head2
        elif phase == 'head3':
            self.classifier.freeze_head1_layers()
            self.classifier.freeze_head2_layers()
            self.classifier.unfreeze_head3_layers()
            optimizer = self.optimizer_head3
            scheduler = self.scheduler_head3
        else:
            raise ValueError(f"Unknown phase: {phase}")

        self.classifier.train()

        for epoch in range(epochs_for_phase):
            total_loss = 0
            num_batches = 0

            pbar = tqdm(train_loader, desc=f"Phase {phase.upper()} - Epoch {epoch+1}/{epochs_for_phase}")
            for features, labels, _ in pbar:
                features = features.to(self.config.device)
                labels = labels.to(self.config.device)

                # Apply class mask
                masked_labels = labels * class_mask.to(self.config.device)

                optimizer.zero_grad()

                # Forward pass با adjacency matrix برای GCN (فقط در phase head3)
                if phase == 'head3' and self.adjacency_matrix_graph is not None:
                    output1, output2, output_final = self.classifier(
                        features,
                        adjacency_matrix=self.adjacency_matrix_graph,
                        training_phase=phase
                    )
                else:
                    output1, output2, output_final = self.classifier(features, training_phase=phase)

                # Calculate loss based on phase
                if phase == 'head1':
                    head1_targets, _ = self.create_hierarchical_targets(masked_labels)
                    head1_targets = head1_targets.to(self.config.device)
                    focal_loss = self.focal_loss(output1, head1_targets)
                    loss = focal_loss
                elif phase == 'head2':
                    _, head2_targets = self.create_hierarchical_targets(masked_labels)
                    head2_targets = head2_targets.to(self.config.device)
                    focal_loss = self.focal_loss(output2, head2_targets)
                    loss = focal_loss
                elif phase == 'head3':
                    focal_loss = self.focal_loss(output_final, masked_labels)
                    # Add hierarchical consistency loss
                    hierarchy_loss = hierarchical_consistency_loss(
                        output_final,
                        self.ancestor_matrix,
                        lambda_consistency=0.1
                    )
                    loss = focal_loss + hierarchy_loss

                loss.backward()

                # Gradient clipping
                torch.nn.utils.clip_grad_norm_(
                    self.classifier.parameters(),
                    self.config.gradient_clip_norm
                )

                optimizer.step()

                total_loss += loss.item()
                num_batches += 1

                pbar.set_postfix({'loss': f'{loss.item():.4f}'})

            avg_loss = total_loss / num_batches
            print(f"Phase {phase.upper()} - Epoch {epoch+1} - Average Loss: {avg_loss:.4f}")

            # Learning rate scheduling
            scheduler.step(avg_loss)

    def validate_with_optimal_thresholds(self, val_loader, dataset_name="Validation", update_thresholds=False):
        """Validate the model with GCN support and optionally update optimal thresholds"""
        self.classifier.eval()
        total_loss = 0
        all_logits = []
        all_labels = []

        with torch.no_grad():
            for features, labels, _ in tqdm(val_loader, desc=f"Evaluating {dataset_name}"):
                features = features.to(self.config.device)
                labels = labels.to(self.config.device)

                # استفاده از GCN در validation
                if self.adjacency_matrix_graph is not None:
                    output1, output2, output_final = self.classifier(
                        features,
                        adjacency_matrix=self.adjacency_matrix_graph,
                        training_phase='all'
                    )
                else:
                    output1, output2, output_final = self.classifier(features, training_phase='all')

                focal_loss = self.focal_loss(output_final, labels)
                hierarchy_loss = hierarchical_consistency_loss(
                    output_final,
                    self.ancestor_matrix,
                    lambda_consistency=0.1
                )
                loss = focal_loss + hierarchy_loss
                total_loss += loss.item()

                all_logits.append(output_final.cpu())
                all_labels.append(labels.cpu())

        all_logits = torch.cat(all_logits)
        all_labels = torch.cat(all_labels)
        print(f"\n🔍 DEBUG INFO:")
        print(f"  Total samples: {len(all_labels)}")
        print(f"  Labels shape: {all_labels.shape}")
        print(f"  Labels sum per sample: min={all_labels.sum(dim=1).min():.2f}, max={all_labels.sum(dim=1).max():.2f}, mean={all_labels.sum(dim=1).mean():.2f}")
        print(f"  Labels sum per class: min={all_labels.sum(dim=0).min():.2f}, max={all_labels.sum(dim=0).max():.2f}")
        print(f"  Non-zero labels: {(all_labels.sum(dim=1) > 0).sum().item()} / {len(all_labels)}")

        # نمایش تعداد هر کلاس
        class_counts = all_labels.sum(dim=0)
        print(f"\n  Class distribution (top 10):")
        for i in torch.argsort(class_counts, descending=True)[:10]:
            print(f"    {self.idx_to_label[i.item()]}: {class_counts[i].item():.0f}")

        # Convert to probabilities
        all_probs = torch.sigmoid(all_logits)

        # Update optimal thresholds if requested
        if update_thresholds:
            self.optimal_thresholds = find_optimal_thresholds(
                all_labels.numpy(),
                all_probs.numpy(),
                self.num_labels
            )
            print(f"Updated optimal thresholds: min={self.optimal_thresholds.min():.3f}, max={self.optimal_thresholds.max():.3f}")

        # Use optimal thresholds if available
        if self.optimal_thresholds is not None:
            y_pred = torch.zeros_like(all_probs)
            for i, threshold in enumerate(self.optimal_thresholds):
                y_pred[:, i] = (all_probs[:, i] > threshold).float()
        else:
            y_pred = (all_probs > 0.5).float()
        print(f"\n🔍 PREDICTION DEBUG:")
        print(f"  Predictions shape: {y_pred.shape}")
        print(f"  Predictions sum per sample: min={y_pred.sum(dim=1).min():.2f}, max={y_pred.sum(dim=1).max():.2f}, mean={y_pred.sum(dim=1).mean():.2f}")
        print(f"  Non-zero predictions: {(y_pred.sum(dim=1) > 0).sum().item()} / {len(y_pred)}")
        print(f"  Probability stats: min={all_probs.min():.4f}, max={all_probs.max():.4f}, mean={all_probs.mean():.4f}")

        f1, precision, recall = hierarchical_f1_score(
            all_logits, all_labels, self.ancestor_matrix.cpu()
        )

        avg_loss = total_loss / len(val_loader)

        y_pred_np = y_pred.int().numpy()
        y_true_np = all_labels.int().numpy()

        target_names = [self.idx_to_label[i] for i in range(self.num_labels)]

        print(f"\n--- {dataset_name} Classification Report ---")
        report = classification_report(y_true_np, y_pred_np, target_names=target_names, zero_division=0)
        print(report)

        return avg_loss, f1, precision, recall

    def fit_improved(self, train_loader, val_loader, test_loader=None, use_cached_features=True):
        """Train the model with all improvements including GCN"""
        print("\n" + "="*60)
        print("STARTING IMPROVED MULTI-HEAD TRAINING WITH GCN")
        print("Features: Focal Loss + Residual Connections + Hierarchical Consistency")
        print("         + Progressive Training + GCN Label Refinement")
        print("="*60)

        # Create progressive training masks
        if use_cached_features:
            self._current_cache = 'train'

        # Get train dataset to compute class masks
        train_dataset = train_loader.dataset
        high_freq_mask, med_freq_mask, all_classes_mask = self.create_progressive_class_masks(train_dataset)

        # Phase 1: Train Head 1 with high frequency classes
        print("\n--- PROGRESSIVE TRAINING STAGE 1: HIGH FREQUENCY CLASSES ---")
        self.train_phase_progressive(train_loader, 'head1', high_freq_mask, epochs_for_phase=3)

        # Validate after Head 1 training
        if use_cached_features:
            self._current_cache = 'val'
        val_loss, val_f1, val_precision, val_recall = self.validate_with_optimal_thresholds(
            val_loader, "Validation after Head 1 (High Freq)", update_thresholds=True
        )

        # Phase 2: Train Head 1 with medium frequency classes
        if use_cached_features:
            self._current_cache = 'train'
        print("\n--- PROGRESSIVE TRAINING STAGE 2: MEDIUM+ FREQUENCY CLASSES ---")
        self.train_phase_progressive(train_loader, 'head1', med_freq_mask, epochs_for_phase=2)

        # Phase 3: Train Head 1 with all classes
        print("\n--- PROGRESSIVE TRAINING STAGE 3: ALL CLASSES ---")
        self.train_phase_progressive(train_loader, 'head1', all_classes_mask, epochs_for_phase=2)

        # Validate after complete Head 1 training
        if use_cached_features:
            self._current_cache = 'val'
        val_loss, val_f1, val_precision, val_recall = self.validate_with_optimal_thresholds(
            val_loader, "Validation after Complete Head 1", update_thresholds=True
        )

        # Phase 4: Train Head 2 progressively
        if use_cached_features:
            self._current_cache = 'train'
        print("\n--- HEAD 2 PROGRESSIVE TRAINING ---")
        self.train_phase_progressive(train_loader, 'head2', high_freq_mask, epochs_for_phase=2)
        self.train_phase_progressive(train_loader, 'head2', med_freq_mask, epochs_for_phase=2)
        self.train_phase_progressive(train_loader, 'head2', all_classes_mask, epochs_for_phase=2)

        # Validate after Head 2 training
        if use_cached_features:
            self._current_cache = 'val'
        val_loss, val_f1, val_precision, val_recall = self.validate_with_optimal_thresholds(
            val_loader, "Validation after Head 2", update_thresholds=True
        )

        # Phase 5: Train Head 3 (final) progressively with GCN
        if use_cached_features:
            self._current_cache = 'train'
        print("\n--- HEAD 3 (FINAL) PROGRESSIVE TRAINING WITH GCN ---")
        self.train_phase_progressive(train_loader, 'head3', high_freq_mask, epochs_for_phase=3)
        self.train_phase_progressive(train_loader, 'head3', med_freq_mask, epochs_for_phase=3)
        self.train_phase_progressive(train_loader, 'head3', all_classes_mask, epochs_for_phase=4)

        # Final validation with threshold optimization
        if use_cached_features:
            self._current_cache = 'val'
        print("\n--- FINAL VALIDATION WITH THRESHOLD OPTIMIZATION ---")
        val_loss, val_f1, val_precision, val_recall = self.validate_with_optimal_thresholds(
            val_loader, "Final Validation", update_thresholds=True
        )

        # Test evaluation if available
        if test_loader is not None:
            if use_cached_features:
                self._current_cache = 'test'
            test_loss, test_f1, test_precision, test_recall = self.validate_with_optimal_thresholds(
                test_loader, "Final Test", update_thresholds=False
            )

            print(f"\n{'='*60}")
            print(f"FINAL RESULTS WITH ALL IMPROVEMENTS + GCN")
            print(f"{'='*60}")
            print(f"Validation - F1: {val_f1:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}")
            print(f"Test - F1: {test_f1:.4f}, Precision: {test_precision:.4f}, Recall: {test_recall:.4f}")
            print(f"{'='*60}")

        # Save final model
        self.save_checkpoint(f'improved_multihead_gcn_model_f1_{val_f1:.4f}.pth')
        print(f"✓ Improved Multi-Head Model with GCN saved!")

    def save_checkpoint(self, filename):
        """Save model checkpoint"""
        checkpoint = {
            'classifier_state_dict': self.classifier.state_dict(),
            'optimizer_head1_state_dict': self.optimizer_head1.state_dict(),
            'optimizer_head2_state_dict': self.optimizer_head2.state_dict(),
            'optimizer_head3_state_dict': self.optimizer_head3.state_dict(),
            'label_to_idx': self.label_to_idx,
            'idx_to_label': self.idx_to_label,
            'ancestor_matrix': self.ancestor_matrix,
            'num_labels': self.num_labels,
            'history': self.history,
            'optimal_thresholds': self.optimal_thresholds,
            'adjacency_matrix_graph': self.adjacency_matrix_graph
        }
        torch.save(checkpoint, os.path.join(self.config.checkpoint_dir, filename))

    def load_checkpoint(self, filepath):
        """Load model checkpoint"""
        checkpoint = torch.load(filepath, map_location=self.config.device)
        self.classifier.load_state_dict(checkpoint['classifier_state_dict'])
        self.optimizer_head1.load_state_dict(checkpoint['optimizer_head1_state_dict'])
        self.optimizer_head2.load_state_dict(checkpoint['optimizer_head2_state_dict'])
        self.optimizer_head3.load_state_dict(checkpoint['optimizer_head3_state_dict'])
        self.history = checkpoint.get('history', self.history)
        self.optimal_thresholds = checkpoint.get('optimal_thresholds', None)
        self.adjacency_matrix_graph = checkpoint.get('adjacency_matrix_graph', None)
        print(f"Improved Multi-Head checkpoint with GCN loaded from {filepath}")

def main():
    """Main function to run the improved training with GCN"""
    config = CFG()

    print("="*60)
    print("LOADING DATASETS FOR IMPROVED MULTI-HEAD TRAINING WITH GCN")
    print("="*60)

    with open(config.train_json) as fp:
        train = json.load(fp)
    with open(config.val_json) as fp:
        valid = json.load(fp)

    test = None
    if os.path.exists(config.test_json):
        with open(config.test_json) as fp:
            test = json.load(fp)
        print(f"✓ Test dataset loaded: {len(test)} samples")
    else:
        print(f"⚠ Test dataset not found at: {config.test_json}")
        print(f"  Training will proceed without test evaluation")

    train_df = pd.DataFrame(train)
    valid_df = pd.DataFrame(valid)
    test_df = pd.DataFrame(test) if test is not None else None

    print(f"✓ Train samples: {len(train_df)}")
    print(f"✓ Validation samples: {len(valid_df)}")
    if test_df is not None:
        print(f"✓ Test samples: {len(test_df)}")
    print("="*60)

    if 'caption' in train_df.columns:
        print("\n✓ Caption column found - will be used in training")
    else:
        print("\n⚠ No caption column found - using text only")

    print("\n" + "="*60)
    print("INITIALIZING IMPROVED MULTI-HEAD MLP WITH GCN")
    print("="*60)

    # اضافه کردن train_df به classifier برای ساخت label graph
    classifier = ImprovedMemeClassifier(config, train_df=train_df)

    # Create datasets
    train_dataset = MemeDataset(
        train_df, config.train_img_dir, classifier.clip_processor,
        classifier.label_to_idx, classifier.ancestor_matrix,
        use_caption=config.use_caption,
        caption_separator=config.caption_separator
    )

    val_dataset = MemeDataset(
        valid_df, config.val_img_dir, classifier.clip_processor,
        classifier.label_to_idx, classifier.ancestor_matrix,
        use_caption=config.use_caption,
        caption_separator=config.caption_separator
    )

    test_dataset = None
    if test_df is not None:
        test_dataset = MemeDataset(
            test_df, config.test_img_dir, classifier.clip_processor,
            classifier.label_to_idx, classifier.ancestor_matrix,
            use_caption=config.use_caption,
            caption_separator=config.caption_separator
        )

    # Pre-compute features for all datasets
    print("\n" + "="*60)
    print("PRE-COMPUTING FEATURES (This will speed up training)")
    print("="*60)

    classifier.precompute_features(train_dataset, 'train', batch_size=config.batch_size)
    classifier.precompute_features(val_dataset, 'val', batch_size=config.batch_size)

    if test_dataset is not None:
        classifier.precompute_features(test_dataset, 'test', batch_size=config.batch_size)

    # Create data loaders with cached features
    train_loader = DataLoader(
        train_dataset, batch_size=config.batch_size,
        shuffle=True, collate_fn=classifier.collate_fn_cached,
        num_workers=0
    )

    val_loader = DataLoader(
        val_dataset, batch_size=config.batch_size,
        shuffle=False, collate_fn=classifier.collate_fn_cached,
        num_workers=0
    )

    test_loader = None
    if test_dataset is not None:
        test_loader = DataLoader(
            test_dataset, batch_size=config.batch_size,
            shuffle=False, collate_fn=classifier.collate_fn_cached,
            num_workers=0
        )

    print("\n" + "="*60)
    print("STARTING IMPROVED TRAINING WITH GCN")
    print("Architecture:")
    print("  Phase 1: Train Head 1 (Ethos/Pathos/Logos) progressively")
    print("  Phase 2: Train Head 2 (5 mid-level categories) progressively")
    print("  Phase 3: Train Head 3 + GCN (22 labels with label graph refinement)")
    print("="*60)

    # Start improved training with GCN
    classifier.fit_improved(train_loader, val_loader, test_loader, use_cached_features=True)

    print("\n" + "="*60)
    print("IMPROVED MULTI-HEAD TRAINING WITH GCN COMPLETED!")
    print("✓ Focal Loss: Better handling of class imbalance")
    print("✓ Residual Connections: Improved gradient flow")
    print("✓ Hierarchical Consistency: Respects label hierarchy")
    print("✓ Progressive Training: Gradual learning from frequent to rare classes")
    print("✓ Optimal Thresholds: Per-class threshold optimization")
    print("✓ GCN Refinement: Label co-occurrence (PMI) + semantic similarity")
    print("✓ Training Improvements: LR scheduling, gradient clipping, early stopping")
    print("="*60)

if __name__ == "__main__":
    main()

LOADING DATASETS FOR IMPROVED MULTI-HEAD TRAINING WITH GCN
✓ Test dataset loaded: 1000 samples
✓ Train samples: 7000
✓ Validation samples: 500
✓ Test samples: 1000

✓ Caption column found - will be used in training

INITIALIZING IMPROVED MULTI-HEAD MLP WITH GCN
Loading RoBERTa model...


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

Phase HEAD3 - Epoch 4 - Average Loss: 0.0591

--- FINAL VALIDATION WITH THRESHOLD OPTIMIZATION ---


Evaluating Final Validation: 100%|██████████| 16/16 [00:09<00:00,  1.76it/s]



🔍 DEBUG INFO:
  Total samples: 500
  Labels shape: torch.Size([500, 31])
  Labels sum per sample: min=4.00, max=16.00, mean=7.10
  Labels sum per class: min=4.00, max=500.00
  Non-zero labels: 500 / 500

  Class distribution (top 10):
    Persuasion: 500
    Ethos: 410
    Pathos: 364
    Other: 364
    Ad Hominem: 340
    Smears: 257
    Justification: 138
    Loaded Language: 135
    Logos: 129
    Name calling/Labeling: 122
Updated optimal thresholds: min=0.174, max=0.877

🔍 PREDICTION DEBUG:
  Predictions shape: torch.Size([500, 31])
  Predictions sum per sample: min=5.00, max=24.00, mean=10.85
  Non-zero predictions: 500 / 500
  Probability stats: min=0.0124, max=0.9965, mean=0.3326

--- Final Validation Classification Report ---
                                                     precision    recall  f1-score   support

                                         Ad Hominem       0.75      0.92      0.83       340
                        Appeal to (Strong) Emotions       0.22     

Evaluating Final Test: 100%|██████████| 32/32 [00:17<00:00,  1.88it/s]



🔍 DEBUG INFO:
  Total samples: 1000
  Labels shape: torch.Size([1000, 31])
  Labels sum per sample: min=4.00, max=16.00, mean=7.39
  Labels sum per class: min=10.00, max=1000.00
  Non-zero labels: 1000 / 1000

  Class distribution (top 10):
    Persuasion: 1000
    Ethos: 850
    Pathos: 750
    Other: 750
    Ad Hominem: 687
    Smears: 504
    Loaded Language: 306
    Justification: 288
    Logos: 284
    Transfer: 274

🔍 PREDICTION DEBUG:
  Predictions shape: torch.Size([1000, 31])
  Predictions sum per sample: min=6.00, max=22.00, mean=12.21
  Non-zero predictions: 1000 / 1000
  Probability stats: min=0.0072, max=0.9982, mean=0.3449

--- Final Test Classification Report ---
                                                     precision    recall  f1-score   support

                                         Ad Hominem       0.72      0.97      0.82       687
                        Appeal to (Strong) Emotions       0.22      0.52      0.31        56
                                